# 01 — Exploratory data analysis (Flickr8k)

Exploration only: every computation calls the `captioner` package. Run `make data` first.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt

from captioner.config import load_config
from captioner.data.prepare import load_prepared

cfg = load_config('../configs/base.yaml', overrides={'paths': {'data_dir': '../data', 'artifacts_dir': '../artifacts'}})
prepared = load_prepared(cfg)
df, vocab = prepared.manifest, prepared.vocab
df.head()

In [ ]:
df.groupby('split', observed=True)['image_id'].nunique()

## Caption length (words, train split)

The `max_len = 25` cap (incl. `<start>`/`<end>`) covers the train p99 of 22 words.

In [ ]:
train = df[df['split'] == 'train']
print(train['n_tokens'].describe())
print('p99:', train['n_tokens'].quantile(0.99))
train['n_tokens'].plot.hist(bins=range(0, 40))
plt.xlabel('words per caption'); plt.ylabel('captions'); plt.show()

## Vocabulary

Train-only counts; `min_freq = 5` keeps 2,529 words. The article "a" is the most frequent word (reference defect D2 deleted it).

In [ ]:
counts = Counter(w for c in train['caption_clean'] for w in str(c).split())
print('distinct train words:', len(counts))
print('kept at min_freq=5:', vocab.n_words)
counts.most_common(20)

In [ ]:
coverage = sum(c for w, c in counts.items() if c >= vocab.min_freq) / sum(counts.values())
print(f'token coverage of the kept vocabulary: {coverage:.2%}')

## Image sizes

In [ ]:
df.drop_duplicates('image_id')[['width', 'height']].describe()